# Getting Started: A Complete Mechanistic Interpretability Investigation

This notebook walks through a **complete mechinterp workflow** from scratch. We'll:

1. Load a model and understand its predictions
2. Use the logit lens to see intermediate predictions
3. Track how the residual stream builds up the answer
4. Find important heads via ablation
5. Analyze what those heads attend to
6. Use gradient-based attribution as a sanity check
7. Study the neurons involved

The task: **factual recall** — "The Eiffel Tower is located in the city of" → " Paris"

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import vis

# Load GPT-2 Small
model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads/layer, d_model={model.cfg.d_model}")

/Users/danielpcox/projects/irtk/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████| 148/148 [00:00<00:00, 1766.95it/s, Materializing param=transformer.wte.weight]


Model: 12 layers, 12 heads/layer, d_model=768


## Step 1: Understand the Model's Prediction

First, let's see what the model predicts and how confident it is.

In [ ]:
prompt = "The Eiffel Tower is located in the city of"
tokens = model.to_tokens(prompt)
token_labels = [tokenizer.decode([t]) for t in np.array(tokens)]

# Run the model
logits, cache = model.run_with_cache(tokens)

# Top predictions at the last position
probs = jax.nn.softmax(logits[-1])
top5 = jnp.argsort(probs)[::-1][:5]

print(f"Prompt: {prompt!r}")
print(f"Tokens: {token_labels}")
print(f"\nTop 5 predictions:")
for idx in top5:
    token_str = tokenizer.decode([int(idx)])
    print(f"  {token_str!r:>10s}: {float(probs[idx]):.1%}")

# Store the target for later
target_token = int(top5[0])
target_str = tokenizer.decode([target_token])
print(f"\nTarget token: {target_str!r} (id={target_token})")

## Step 2: Logit Lens — When Does the Model "Know" the Answer?

The **logit lens** projects each layer's residual stream through the unembedding to see what the model would predict at that point.

In [ ]:
from irtk.logit_lens import logit_lens, logit_lens_top_k

# Get logit lens predictions
ll_logits = logit_lens(model, tokens, return_probs=True)

# Plot the probability of the correct answer at each layer
target_probs = ll_logits[:, -1, target_token]  # [n_layers+1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(len(target_probs)), target_probs, 'o-', markersize=8)
ax.set_xlabel('Layer (0=embed, 1-12=after each block)')
ax.set_ylabel(f'P({target_str!r})')
ax.set_title('Logit Lens: When does the model predict the answer?')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, max(target_probs) * 1.2)
plt.tight_layout()
plt.show()

# Top predictions at each layer
top_k = logit_lens_top_k(model, tokens, k=3)
print("Top prediction at each layer (last position):")
for layer_idx, layer_preds in enumerate(top_k):
    preds_at_last = layer_preds[-1]
    top_str = tokenizer.decode([preds_at_last[0][0]])
    top_prob = preds_at_last[0][1]
    label = 'embed' if layer_idx == 0 else f'L{layer_idx-1}'
    print(f"  {label:>6s}: {top_str!r} ({top_prob:.1%})")

## Step 3: Residual Stream Analysis

The residual stream is the sum of all components' outputs. Let's see how each component contributes to the final logit.

In [ ]:
from irtk.residual_stream import (
    residual_norm_by_layer,
    residual_direction_analysis,
    prediction_rank_trajectory,
)

# How does the residual stream norm grow?
norms = residual_norm_by_layer(cache, pos=-1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(range(len(norms)), norms, 'o-')
ax1.set_xlabel('Layer')
ax1.set_ylabel('Residual Stream Norm')
ax1.set_title('Residual Stream Norm Growth')
ax1.grid(True, alpha=0.3)

# Where does the target token rank at each layer?
ranks = prediction_rank_trajectory(model, tokens, target_token)
ax2.plot(range(len(ranks)), ranks, 'o-', color='tab:orange')
ax2.set_xlabel('Layer')
ax2.set_ylabel(f'Rank of {target_str!r}')
ax2.set_title('Prediction Rank Trajectory')
ax2.invert_yaxis()  # Lower rank = better
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Decompose: which components push toward the answer?
W_U = model.unembed.W_U
target_dir = W_U[:, target_token]
analysis = residual_direction_analysis(model, cache, target_dir, pos=-1)

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(analysis['labels'])), analysis['components'])
ax.set_xticks(range(len(analysis['labels'])))
ax.set_xticklabels(analysis['labels'], rotation=90, fontsize=7)
ax.set_title(f'Component Contributions to {target_str!r} Logit')
ax.set_ylabel('Projection onto logit direction')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 4: Find Important Heads via Ablation

Now let's systematically test which attention heads matter for this prediction.

In [ ]:
from irtk.experiments import sweep_ablations, find_circuit

metric = lambda logits: float(logits[-1, target_token])
baseline = metric(logits)
print(f"Baseline logit for {target_str!r}: {baseline:.3f}")

# Ablate each head
result = sweep_ablations(model, [tokens], metric, method="zero", component="heads")
effect = result.values - baseline  # negative = ablation hurts

fig, ax = vis.plot_head_summary(
    effect,
    title=f"Effect of Ablating Each Head on {target_str!r} Logit",
    cmap="RdBu_r",
    vmin=-3, vmax=3,
)
plt.show()

# Find the circuit
circuit = find_circuit(model, [tokens], metric, threshold=0.05)
print(f"\nCircuit heads (>{5}% effect):")
for l, h, imp in circuit.values['circuit_heads'][:10]:
    direction = 'hurts' if (effect[l, h] < 0) else 'helps'
    print(f"  L{l}H{h}: {imp:.0%} change ({direction} when ablated, effect={effect[l,h]:+.3f})")

## Step 5: Analyze Attention Patterns of Important Heads

For the most important heads, let's look at what they attend to.

In [ ]:
from irtk.attention_utils import (
    top_attended_tokens, entropy, attention_head_summary,
)

# Top 4 most impactful heads (by absolute effect)
important_heads = sorted(
    [(l, h, abs(effect[l, h])) for l in range(model.cfg.n_layers) for h in range(model.cfg.n_heads)],
    key=lambda x: x[2], reverse=True
)[:4]

fig, axes = plt.subplots(1, len(important_heads), figsize=(5*len(important_heads), 5))

for ax, (l, h, imp) in zip(axes, important_heads):
    pattern = np.array(cache[("pattern", l)][h])
    im = ax.imshow(pattern, cmap='viridis', aspect='auto')
    ax.set_title(f"L{l}H{h} (effect={effect[l,h]:+.2f})")
    if len(token_labels) <= 15:
        ax.set_xticks(range(len(token_labels)))
        ax.set_xticklabels(token_labels, rotation=90, fontsize=6)
        ax.set_yticks(range(len(token_labels)))
        ax.set_yticklabels(token_labels, fontsize=6)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Attention Patterns of Most Impactful Heads', fontsize=14)
plt.tight_layout()
plt.show()

# What do they attend to from the last position?
print("\nAttention from last position:")
for l, h, imp in important_heads:
    top = top_attended_tokens(cache, l, h, query_pos=-1, k=5)
    top_strs = [f"{token_labels[pos]}({w:.2f})" for pos, w in top]
    print(f"  L{l}H{h}: {', '.join(top_strs)}")

## Step 6: Gradient-Based Sanity Check

Let's verify our findings with gradient-based attribution. Which input tokens are most important?

In [ ]:
from irtk.gradients import gradient_x_input, integrated_gradients, token_attribution

# Integrated gradients (most principled)
ig_attr = integrated_gradients(model, tokens, target_token, n_steps=30)

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['tab:red' if v > np.percentile(ig_attr, 80) else 'tab:blue' for v in ig_attr]
ax.bar(range(len(ig_attr)), ig_attr, color=colors)
ax.set_xticks(range(len(token_labels)))
ax.set_xticklabels(token_labels, rotation=45, ha='right')
ax.set_title(f'Integrated Gradients: Which tokens matter for {target_str!r}?')
ax.set_ylabel('Attribution')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Most important input tokens (by integrated gradients):")
for i in np.argsort(ig_attr)[::-1][:5]:
    print(f"  {token_labels[i]:>10s}: {ig_attr[i]:.4f}")

## Step 7: Neuron-Level Analysis

Let's look at which MLP neurons in the important layers contribute most to the answer.

In [ ]:
from irtk.neurons import neuron_attribution, neuron_to_logit

# Check a few important layers
important_layers = sorted(set(l for l, h, _ in important_heads))[-3:]

for layer in important_layers:
    attr = neuron_attribution(model, cache, layer, target_token)
    top_neurons = np.argsort(np.abs(attr))[::-1][:5]
    
    print(f"\nLayer {layer} — top neurons for {target_str!r}:")
    for n in top_neurons:
        promoted, _ = neuron_to_logit(model, layer, int(n), k=3)
        promoted_strs = [tokenizer.decode([t]) for t, _ in promoted]
        print(f"  neuron {n}: attr={attr[n]:+.4f}, promotes: {promoted_strs}")

## Step 8: Direct Logit Attribution — Per-Head Breakdown

Finally, let's see each head's direct contribution to the target logit.

In [ ]:
from irtk.circuits import direct_logit_attribution

head_attrs = direct_logit_attribution(model, cache, token=target_token)

fig, ax = vis.plot_head_summary(
    head_attrs,
    title=f"Direct Logit Attribution for {target_str!r}",
    cmap="RdBu_r",
    vmin=-2, vmax=2,
)
plt.show()

print(f"Top 5 heads promoting {target_str!r}:")
flat = head_attrs.flatten()
for i in np.argsort(flat)[::-1][:5]:
    l = i // model.cfg.n_heads
    h = i % model.cfg.n_heads
    print(f"  L{l}H{h}: {head_attrs[l,h]:+.3f}")

print(f"\nTop 5 heads suppressing {target_str!r}:")
for i in np.argsort(flat)[:5]:
    l = i // model.cfg.n_heads
    h = i % model.cfg.n_heads
    print(f"  L{l}H{h}: {head_attrs[l,h]:+.3f}")

## Summary

In this investigation, we:

1. **Verified the prediction**: The model correctly predicts the next token
2. **Logit lens**: Showed which layer the answer "crystallizes" at
3. **Residual stream**: Tracked how components build up the answer logit
4. **Ablation**: Identified the critical attention heads
5. **Attention patterns**: Saw what those heads attend to
6. **Gradient attribution**: Confirmed which input tokens are most important
7. **Neuron analysis**: Found specific neurons promoting the answer
8. **Direct logit attribution**: Measured each head's direct contribution

### The IRTK Toolkit at a Glance

| Module | Purpose |
|--------|--------|
| `irtk.HookedTransformer` | Load models, run with cache/hooks |
| `irtk.logit_lens` | Per-layer predictions |
| `irtk.residual_stream` | Residual stream analysis |
| `irtk.circuits` | OV/QK circuits, composition, logit attribution |
| `irtk.patching` | Activation patching, path patching |
| `irtk.experiments` | Batch experiments, circuit discovery |
| `irtk.attention_utils` | Attention entropy, causal tracing |
| `irtk.gradients` | Gradient-based attribution |
| `irtk.neurons` | Neuron-level analysis |
| `irtk.probes` | Linear probes |
| `irtk.sae` | Sparse autoencoders |
| `irtk.weight_processing` | Fold LayerNorm, center weights |
| `irtk.model_diff` | Compare two models |
| `irtk.vis` | Visualization helpers |
| `irtk.datasets` | IOI, repeated tokens, etc. |

See `01_api_cheatsheet.ipynb` for the full API reference!